# Kompilace modelu pro Hailo-8 akcelerátor
Tento návod popisuje proces konverze YOLOv8 modelu do Hailo HEF formátu pro inference na Hailo-8 akcelerátoru.

## Možnosti kompilace modelu
1. **COCO Dataset**
- Je obří akademický dataset (stovky tisíc obrázků), který obsahuje 80 tříd (od aut a lidí přes žirafy, slony až po zubní kartáčky a hotdogy).
- Skvělý pro obecné "hračkové" a standardní modely, které mají poznávat svět kolem sebe.
2. **Vlatní Dataset**
- Obrázky z vlastních dat.
- Je to "Svatý grál" edge AI. Kompilátor díky nim přesně vidí, jaké barvy, jaký šum z kamery a jaké světlo bude sítí reálně protékat. Ořízne a zkalibruje 8-bitové váhy s naprosto chirurgickou přesností přímo pro vaše podmínky.
- Absolutní nutnost pro profesionální, specializované a průmyslové nasazení, kde se hraje o každé procento přesnosti.
3. **Náhodná data:**
- Pouze rychlý test softwaru. Nepřesný

## Předpoklady
- Nainstalované Hailo Dataflow Compiler (DFC) pro Hailo-8/8L (odkaz:`https://hailo.ai/developer-zone/software-downloads/`)
- Nainstalovaný hailo-model-zoo pro hailo Hailo-8/8L (odkaz:`https://hailo.ai/developer-zone/software-downloads/`)
- Linux OS x86 (Ubuntu,... ) nebo Google Colab
- Python 3.10 prostředí

**Vytvoření a nastavení prostředí na Ubuntu PC (v terminálu)**:
```bash
python3.10 -m venv hailo_env
source hailo_env/bin/activate
pip install --upgrade pip
pip install ultralytics
pip install -U pip setuptools wheel
pip install hailo_dataflow_compiler-3.33.0-py3-none-linux_x86_64.whl
pip install ./hailo_model_zoo-2.17.1-py3-none-any.whl
```

V tom návodu bude použit pro kompilaci Google Colab, protože umí zajistit dostatečně výkonné GPU T4 pro správné provedení kompilace.

###Nastavení python pro správnou verzi prostředí (pouze google colab)

In [14]:
# Instalace Pythonu 3.10 a potřebných vývojářských balíčků
!apt-get update -q
!apt-get install python3.10 python3.10-dev python3.10-distutils -y -q

# Nastavení Pythonu 3.10 jako hlavní verze systému
!update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1
!update-alternatives --set python3 /usr/bin/python3.10

# Stažení a instalace čistého správce balíčků pip pro tuto verzi
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10

# Rychlá kontrola - mělo by to vypsat Python 3.10.x
!python3 --version

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lis

Instalace potřebných knihoven a balíčků. DFC a Hailo Model Zoo se dají stáhnout jako package pro Python z oficiálních stránek Hailo (https://hailo.ai/developer-zone/software-downloads/?product=ai_accelerators&device=hailo_8_8l). Při stažení si zároveň musíme dát pozor, že ty packages stahujeme pro správnou verzi Hailo akcelerátoru (např.: Pro můj device to je hailo-8/8L.)

In [15]:
!python3 -m pip install --upgrade pip
!python3 -m pip install ultralytics
!python3 -m pip install -U pip setuptools wheel
!apt-get update
!apt-get install graphviz libgraphviz-dev -y
!python3 -m pip install ./hailo_dataflow_compiler-3.33.0-py3-none-linux_x86_64.whl
!python3 -m pip install ./hailo_model_zoo-2.17.1-py3-none-any.whl

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

## Kompilace pomocí COCO datasetu
### Stažení COCO datasetu s obrázky

**Zadejte do terminálu:**
```bash
mkdir -p coco_full && cd coco_full
curl -s http://images.cocodataset.org/zips/val2017.zip -o val2017.zip
unzip -q val2017.zip && mv val2017/* . && rm -rf val2017 val2017.zip
cd ..
```



### Spuštění kompilace (parsing, optimisation, resource allocation, HEF generation)

Pro yolov8s model

In [16]:
%env USER=root # to se nastavuje jen kvůli google prostředí
!hailomz compile yolov8s --hw-arch hailo8 --calib-path ./coco_full --classes 80 --performance

Traceback (most recent call last):
  File "/usr/local/bin/hailomz", line 6, in <module>
    sys.exit(main())
  File "/usr/local/lib/python3.10/dist-packages/hailo_model_zoo/main.py", line 122, in main
    run(args)
  File "/usr/local/lib/python3.10/dist-packages/hailo_model_zoo/main.py", line 101, in run
    from hailo_model_zoo.main_driver import compile, evaluate, optimize, parse, profile
  File "/usr/local/lib/python3.10/dist-packages/hailo_model_zoo/main_driver.py", line 10, in <module>
    from hailo_sdk_client import ClientRunner, InferenceContext
ModuleNotFoundError: No module named 'hailo_sdk_client'


Když máme model optimalizovaný pomocí GPU, tak už jenom stačí dokončit zbylé fáze resource allocation a HEF compilation na PC. Pomocí: "hailo compiler yolov8s.har --hw-arch hailo8"

In [17]:
from google.colab import files
files.download('yolov8s.har')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Možnost kompilace vlastního modelu přes vlastní kalibrační dataset

### Export modelu z Ultralytics

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8m.pt')
model.export(format='onnx', simplify=True, opset=11)

### Parsing ONNX modelu


In [ ]:
hailo parser onnx yolov8m.onnx --hw-arch hailo8 \
  --end-node-names /model.22/cv2.0/cv2.0.2/Conv \
                   /model.22/cv3.0/cv3.0.2/Conv \
                   /model.22/cv2.1/cv2.1.2/Conv \
                   /model.22/cv3.1/cv3.1.2/Conv \
                   /model.22/cv2.2/cv2.2.2/Conv \
                   /model.22/cv3.2/cv3.2.2/Conv \
                   /model.22/Concat_3

**Co se stane:**
- Parser analyzuje strukturu ONNX modelu
- Díky end-nodes pochopí, kde končí konvoluce a kde začíná NMS.
- Detekuje YOLOv8 architekturu
- Vytvoří `yolov8m.har` soubor
- Automaticky přidá NMS post-processing (pokud potvrdíte doporučení)
- výstupem je har soubor

### Optimalizace
Vytvoříme si složku, kam zkopírujeme své kalibrační obrázky.
```bash
mkdir calibration_dataset
cp /cesta/k/obrazkum/*.jpg calibration_dataset/
```



In [ ]:
!hailo optimize yolov8m.har --calib-set-path calibration_dataset/ 


**Doporučení pro kalibrační data:**
- 100-500 reprezentativních obrázků
- Formáty: JPG, PNG
- Obrázky podobné produkčním datům (dopravní scény)
- Různé světelné podmínky a úhly

**Co se stane:**
- Model je kvantizován z FP32 na INT8
- Použijí se kalibrační data pro zachování přesnosti
- Vytvoří se optimalizovaný HAR soubor

- Pro testování: použijte `--use-random-calib-set`

### Kompilace a vytvoření HEF souboru

In [ ]:
hailo compiler yolov8m_optimized.har \
  --output-dir ./

Výstupem bude hef soubor.

**Čas:** 15-45 minut pro YOLOv8m (závisí na CPU a RAM)

## Další Kroky
Po vytvoření HEF souboru můžete:

1. **Testovat model:**
```bash
hailortcli run yolov8m.hef
```
2. **Integrovat do aplikace** pomocí HailoRT API (Python/C++)
   
3. **Benchmark výkonu:**
```bash
hailortcli benchmark yolov8m.hef
```

**Analýza HAR souboru**
```bash
hailo visualizer yolov8m.har
```

**Kontrola HEF souboru**
```bash
hailortcli scan
```

## Odkazy
- [Hailo Documentation](https://hailo.ai/developer-zone/documentation/)
- [Hailo Model Zoo](https://github.com/hailo-ai/hailo_model_zoo)
- [YOLOv8 Ultralytics](https://docs.ultralytics.com/)


